# AASTU Mini Semantic Search
Week 2: Data, APIs & Embeddings. Pipeline: load → clean → prep text → embed → store in Chroma → query.

## Clone the repo and enter the folder:

In [5]:
# git clone <https://github.com/kedirzeynu/week-02-aastu-semantic-search.git>
# $cd week-02-aastu-semantic-search

In [1]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/<your-username>/aastu-semantic-search.git"

if "google.colab" in sys.modules:                      # running in Colab
    if not Path("aastu-semantic-search").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("aastu-semantic-search")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

ROOT = Path.cwd()                                      # find the repo root
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "week-02" / "src"))
print("Project root:", ROOT)

Project root: /home/kedir/week-02-aastu-semantic-search


In [ ]:
import numpy as np
import pandas as pd

from load_data import load_faqs, load_courses, first_look
from cleaning import clean_faqs, quality_report
from numpy_stats import summarize
from text_prep import prepare_text_columns, chunk_words
from embeddings import get_embedder, embed_texts
from similarity import cosine, top_k
from search import build_collection, search, print_hits, slide_style_demo

ModuleNotFoundError: No module named 'load_data'

In [ ]:
raw = load_faqs()
first_look(raw)

In [ ]:
quality_report(raw, "BEFORE")
faqs = clean_faqs(raw)
quality_report(faqs, "AFTER")
faqs.head()

In [ ]:
summarize(faqs["views"])

In [ ]:
faqs = prepare_text_columns(faqs)
print(faqs[["text_clean", "n_tokens"]].head(3))
handbook_words = " ".join(faqs["answer"]).split()
chunks = chunk_words(handbook_words, size=60, overlap=12)
print("words:", len(handbook_words), "| chunks:", [len(c) for c in chunks])

In [ ]:
offline_json_demo()

In [ ]:
embedder = get_embedder()
vectors = embed_texts(embedder, faqs["text"])
print(vectors.shape)          # (21, 384)

In [ ]:
q = embed_texts(embedder, ["I forgot my password"])[0]
idx, scores = top_k(vectors, q, k=3)
for i, s in zip(idx, scores):
    print(f"{s:.2f}  {faqs.iloc[i]['question']}")

In [ ]:
col = build_collection(faqs, embedder)
for query in ["I forgot my password", "When is registration?", "Where can I study quietly?"]:
    print_hits(query, search(col, embedder, query, k=3))

In [ ]:
print_hits("how do I pay", search(col, embedder, "how do I pay", k=3, where={"category": "Fees"}))

In [ ]:
try:
    hits = slide_style_demo(faqs["text"].tolist(), [f"s{i}" for i in faqs["id"]])
    print(hits["documents"][0][0])
except Exception as error:
    print("Needs internet for the model download:", type(error).__name__)